# 0.1. Ustawienie ścieżek

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve() / "src"))

from config.config import setup

cfg = setup()

HAR_ROOT = cfg["HAR_ROOT"]
MODELS_DIR = cfg["MODELS_DIR"]
RESULTS_DIR = cfg["RESULTS_DIR"]

# 0.2. Wgranie danych z datasetu

In [2]:
from data.loader import load_har_data

X_train, y_train, groups_train, X_test, y_test, groups_test, feature_names = (
    load_har_data(HAR_ROOT)
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

X_train: (7352, 561) | X_test: (2947, 561)


# 0.3. Konfiguracja danych treningowych i parametrów eksperymentu (CV, metryka, wyniki)

In [3]:
import numpy as np
from sklearn.model_selection import GroupKFold
from src.tuning.grid_search_cv import run_grid
from src.config.constants import N_JOBS

y_train_arr = np.asarray(y_train)
groups_train_arr = np.asarray(groups_train)

CV = GroupKFold(n_splits=5)
SCORING = "matthews_corrcoef"
ALL_SUMMARIES = []

# 1. Sprawdzenie rozkładu osób w foldach GroupKFold(5):

In [4]:
print("Sprawdzenie rozkładu osób w foldach GroupKFold(5):")
for i, (tr, va) in enumerate(
    CV.split(X_train, y_train_arr, groups=groups_train_arr), 1
):
    train_subj = np.unique(groups_train_arr[tr]).astype(int).tolist()
    val_subj = np.unique(groups_train_arr[va]).astype(int).tolist()
    print(
        f"fold {i}: train_subj={str(train_subj):<65}  |  val_subj={str(val_subj):<20}  ({len(va)} okien)"
    )

Sprawdzenie rozkładu osób w foldach GroupKFold(5):
fold 1: train_subj=[1, 3, 5, 6, 7, 8, 11, 16, 17, 21, 22, 23, 26, 27, 28, 29, 30]     |  val_subj=[14, 15, 19, 25]      (1420 okien)
fold 2: train_subj=[1, 3, 5, 7, 8, 11, 14, 15, 17, 19, 23, 25, 26, 27, 28, 29, 30]    |  val_subj=[6, 16, 21, 22]       (1420 okien)
fold 3: train_subj=[1, 5, 6, 7, 8, 14, 15, 16, 19, 21, 22, 23, 25, 27, 28, 29, 30]    |  val_subj=[3, 11, 17, 26]       (1417 okien)
fold 4: train_subj=[3, 5, 6, 8, 11, 14, 15, 16, 17, 19, 21, 22, 25, 26, 27, 28, 29]   |  val_subj=[1, 7, 23, 30]        (1410 okien)
fold 5: train_subj=[1, 3, 6, 7, 11, 14, 15, 16, 17, 19, 21, 22, 23, 25, 26, 30]       |  val_subj=[5, 8, 27, 28, 29]    (1685 okien)


# 2. DummyClassifier – strojenie hiperparametrów

In [5]:
from models.pipelines import pipe_dummy

param_grid_dummy = {
    "clf__strategy": ["stratified", "most_frequent", "uniform"],
}

grid_dummy, s = run_grid(
    "dummy",
    pipe_dummy,
    param_grid_dummy,
    X=X_train,
    y=y_train_arr,
    groups=groups_train_arr,
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
    cv=CV,
    scoring=SCORING,
    n_jobs=N_JOBS,
)

ALL_SUMMARIES.append(s)

Fitting 5 folds for each of 3 candidates, totalling 15 fits

[dummy]	Best MCC = 0.0034		|		3 kombinacji × 5 foldów = 15 fitów		|		czas = 0.3s
[dummy]	Best_params: {'clf__strategy': 'uniform'}


# 3. LogisticRegression

In [6]:
from models.pipelines import pipe_logreg

param_grid_logreg = [
    {
        "clf__C": [0.01, 0.1, 1.0, 10.0],
        "clf__solver": ["lbfgs", "newton-cg"],
    }
]

grid_logreg, s = run_grid(
    "logreg",
    pipe_logreg,
    param_grid_logreg,
    X=X_train,
    y=y_train_arr,
    groups=groups_train_arr,
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
    cv=CV,
    scoring=SCORING,
    n_jobs=N_JOBS,
)

ALL_SUMMARIES.append(s)

Fitting 5 folds for each of 8 candidates, totalling 40 fits

[logreg]	Best MCC = 0.9227		|		8 kombinacji × 5 foldów = 40 fitów		|		czas = 18.6s
[logreg]	Best_params: {'clf__C': 1.0, 'clf__solver': 'newton-cg'}


# 4. LinearSVC

In [7]:
from models.pipelines import pipe_linear_svc

param_grid_lsvc = {
    "clf__C": [0.01, 0.1, 1.0, 10.0],
    "clf__loss": ["squared_hinge"],
}

grid_lsvc, s = run_grid(
    "linear_svc",
    pipe_linear_svc,
    param_grid_lsvc,
    X=X_train,
    y=y_train_arr,
    groups=groups_train_arr,
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
    cv=CV,
    scoring=SCORING,
    n_jobs=N_JOBS,
)

ALL_SUMMARIES.append(s)

Fitting 5 folds for each of 4 candidates, totalling 20 fits

[linear_svc]	Best MCC = 0.9258		|		4 kombinacji × 5 foldów = 20 fitów		|		czas = 203.1s
[linear_svc]	Best_params: {'clf__C': 0.1, 'clf__loss': 'squared_hinge'}


# 5. SVC (RBF)

In [8]:
from models.pipelines import pipe_rbf_svc

param_grid_rbf = {
    "clf__C": [0.1, 1.0, 10.0],
    "clf__gamma": ["scale", 0.01, 0.001],
}

grid_rbf, s = run_grid(
    "rbf_svc",
    pipe_rbf_svc,
    param_grid_rbf,
    X=X_train,
    y=y_train_arr,
    groups=groups_train_arr,
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
    cv=CV,
    scoring=SCORING,
    n_jobs=N_JOBS,
)

ALL_SUMMARIES.append(s)

Fitting 5 folds for each of 9 candidates, totalling 45 fits

[rbf_svc]	Best MCC = 0.9177		|		9 kombinacji × 5 foldów = 45 fitów		|		czas = 274.7s
[rbf_svc]	Best_params: {'clf__C': 10.0, 'clf__gamma': 0.001}


# 6. RandomForest

In [9]:
from models.pipelines import pipe_random_forest

param_grid_rf = {
    "clf__n_estimators": [300, 500],
    "clf__max_depth": [None, 20],
    "clf__min_samples_split": [2, 10],
    "clf__max_features": ["sqrt"],
}

grid_rf, s = run_grid(
    "random_forest",
    pipe_random_forest,
    param_grid_rf,
    X=X_train,
    y=y_train_arr,
    groups=groups_train_arr,
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
    cv=CV,
    scoring=SCORING,
    n_jobs=1,
)

ALL_SUMMARIES.append(s)

Fitting 5 folds for each of 8 candidates, totalling 40 fits

[random_forest]	Best MCC = 0.9041		|		8 kombinacji × 5 foldów = 40 fitów		|		czas = 93.7s
[random_forest]	Best_params: {'clf__max_depth': None, 'clf__max_features': 'sqrt', 'clf__min_samples_split': 10, 'clf__n_estimators': 300}


# 7. GaussianNB

In [10]:
from models.pipelines import pipe_gaussian_nb

param_grid_gnb = {
    "clf__var_smoothing": [1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5],
}

grid_gnb, s = run_grid(
    "gaussian_nb",
    pipe_gaussian_nb,
    param_grid_gnb,
    X=X_train,
    y=y_train_arr,
    groups=groups_train_arr,
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
    cv=CV,
    scoring=SCORING,
    n_jobs=N_JOBS,
)

ALL_SUMMARIES.append(s)

Fitting 5 folds for each of 6 candidates, totalling 30 fits

[gaussian_nb]	Best MCC = 0.6911		|		6 kombinacji × 5 foldów = 30 fitów		|		czas = 3.5s
[gaussian_nb]	Best_params: {'clf__var_smoothing': 1e-05}


# 8. Podsumowanie

In [22]:
import pandas as pd

summary_df = pd.DataFrame(ALL_SUMMARIES)
summary_df = summary_df.sort_values("best_score", ascending=False).reset_index(
    drop=True
)
summary_df.to_csv(RESULTS_DIR / "grid_summary.csv", index=False)

timings_df = summary_df[["name", "n_combos", "n_fits", "time_s"]].copy()
timings_df.to_csv(RESULTS_DIR / "timings_grid.csv", index=False)

print("=" * 70)
print("PODSUMOWANIE STROJENIA (CV MCC, GroupKFold(5) po osobach)")
print("=" * 70)
for _, r in summary_df.iterrows():
    print(
        f"{r['name']:<14} | MCC = {r['best_score']:.4f} | t = {r['time_s']:>7.1f}s | "
        f"{r['n_fits']} fitów"
    )
    print(f"               best_params: {r['best_params']}")

PODSUMOWANIE STROJENIA (CV MCC, GroupKFold(5) po osobach)
linear_svc     | MCC = 0.9258 | t =   203.2s | 20 fitów
               best_params: {'clf__C': 0.1, 'clf__loss': 'squared_hinge'}
logreg         | MCC = 0.9227 | t =    18.6s | 40 fitów
               best_params: {'clf__C': 1.0, 'clf__solver': 'newton-cg'}
rbf_svc        | MCC = 0.9177 | t =   274.7s | 45 fitów
               best_params: {'clf__C': 10.0, 'clf__gamma': 0.001}
random_forest  | MCC = 0.9041 | t =    93.7s | 40 fitów
               best_params: {'clf__max_depth': None, 'clf__max_features': 'sqrt', 'clf__min_samples_split': 10, 'clf__n_estimators': 300}
gaussian_nb    | MCC = 0.6911 | t =     3.5s | 30 fitów
               best_params: {'clf__var_smoothing': 1e-05}
dummy          | MCC = 0.0034 | t =     0.3s | 15 fitów
               best_params: {'clf__strategy': 'uniform'}


# 9. Sprawdzenie czy model działa, czy predict() działa

In [21]:
import joblib

best_name = summary_df.iloc[0]["name"]
best_grid = joblib.load(MODELS_DIR / f"grid_{best_name}.joblib")
y_true_sample = y_train_arr[:5]
y_pred_sample = best_grid.best_estimator_.predict(X_train.iloc[:5])
print(f"Najlepszy CV: {best_name}")
print(f"{'y_true[:5]':<15}= {list(y_true_sample)}")
print(f"{'y_pred[:5]':<15}= {list(y_pred_sample)}")
print(f"{'klasy modelu':<15}= {list(best_grid.classes_)}")

Najlepszy CV: linear_svc
y_true[:5]     = ['STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING']
y_pred[:5]     = ['STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING']
klasy modelu   = ['LAYING', 'SITTING', 'STANDING', 'WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS']
